# ☁️ Cloud Fundamentals — Python + GCP

Projeto desenvolvido durante o curso de **Fundamentos de Nuvem** da Alura, adaptado para explorar na prática o uso do Python como ferramenta de acesso e manipulação de dados dentro do ecossistema **Google Cloud Platform (GCP)**.

O fluxo do projeto é:
1. Autenticar no GCP via Google Colab
2. Ler um arquivo JSON do **Cloud Storage**
3. Consultar dados de pedidos no **BigQuery**
4. Identificar pedidos com atraso via SQL
5. Salvar o resultado como nova tabela no BigQuery
6. Fazer upload do CSV de resultado de volta ao Cloud Storage

## 🔐 1. Autenticação no GCP

In [ ]:
from google.colab import auth

# Abre o pop-up de autenticação do Google — necessário para acessar qualquer serviço GCP
auth.authenticate_user()

## ⚙️ 2. Configurações Gerais

Centralizando as variáveis de configuração em um único lugar — assim fica fácil adaptar o projeto para outro projeto GCP ou bucket.

In [ ]:
# Configurações do projeto GCP
PROJECT_ID   = 'alura-studies'
BUCKET_NAME  = 'cloudgc_fundaments'
FILE_NAME    = 'BR.json'

# Destino da tabela no BigQuery
DATASET_ID   = 'olist_ecommerce'
TABLE_PATH   = f'{PROJECT_ID}.{DATASET_ID}.pedidos_atrasados'

## 📦 3. Importações

In [ ]:
import json
from google.cloud import storage
from google.cloud import bigquery

## 🗄️ 4. Leitura de Arquivo no Cloud Storage

Conectando ao GCS e fazendo o download do arquivo `BR.json`, que contém os feriados nacionais do Brasil.

In [ ]:
# Inicializa o cliente do Cloud Storage
client_gcs = storage.Client(project=PROJECT_ID)

# Acessa o bucket e o arquivo desejado
bucket = client_gcs.bucket(BUCKET_NAME)   # ✅ uso do método atual (get_bucket é depreciado)
blob   = bucket.blob(FILE_NAME)

# Faz o download do conteúdo como string
file_content_str = blob.download_as_text()

print(f"Conteúdo do arquivo {FILE_NAME}:")
print(file_content_str)

In [ ]:
# Converte a string JSON para uma lista de dicionários Python
dados_feriados = json.loads(file_content_str)

print(f"Total de feriados carregados: {len(dados_feriados)}")
print("Primeiro feriado:", dados_feriados[0])

## 🔍 5. Consulta no BigQuery — Todos os Pedidos

Conectando ao BigQuery e trazendo todos os pedidos da base Olist para ter uma visão geral dos dados.

In [ ]:
# Inicializa o cliente do BigQuery
client_bq = bigquery.Client(project=PROJECT_ID)

In [ ]:
consulta_pedidos = """
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    order_delivered_customer_date
FROM `alura-studies.olist_dataset.orders`
"""

pedidos = client_bq.query(consulta_pedidos).to_dataframe()

print(f"Total de pedidos: {len(pedidos)}")
pedidos.head()

## ⏰ 6. Consulta no BigQuery — Pedidos com Atraso

Filtrando apenas os pedidos que chegaram **depois** da data estimada e calculando quantos dias de atraso cada um teve.

> **Detalhe importante:** a condição `order_delivered_customer_date > order_estimated_delivery_date` compara data *e hora*. Isso significa que um pedido entregue às 23h59 no mesmo dia estimado passaria pelo filtro, mas o `DATE_DIFF` com granularidade `DAY` retornaria `0`. Por isso adicionamos `AND DATE_DIFF(...) > 0` para garantir que só entram pedidos com atraso real de pelo menos 1 dia.

In [ ]:
consulta_atrasos = """
SELECT
    order_id,
    order_estimated_delivery_date,
    order_delivered_customer_date,
    DATE_DIFF(order_delivered_customer_date, order_estimated_delivery_date, DAY) AS atraso_dias
FROM `alura-studies.olist_dataset.orders`
WHERE
    order_delivered_customer_date IS NOT NULL
    AND order_estimated_delivery_date IS NOT NULL
    AND order_delivered_customer_date > order_estimated_delivery_date
    AND DATE_DIFF(order_delivered_customer_date, order_estimated_delivery_date, DAY) > 0
ORDER BY atraso_dias DESC
"""

df_atraso = client_bq.query(consulta_atrasos).to_dataframe()

print(f"Total de pedidos atrasados: {len(df_atraso)}")
df_atraso.head()

## 💾 7. Salvando Resultados no BigQuery

Criando (se não existir) o dataset `olist_ecommerce` e carregando o DataFrame de atrasos como uma nova tabela.

In [ ]:
# Verifica se o dataset já existe; se não, cria
dataset_ref = client_bq.dataset(DATASET_ID)

try:
    client_bq.get_dataset(dataset_ref)
    print(f"Dataset '{DATASET_ID}' já existe.")
except Exception as e:
    if 'Not found: Dataset' in str(e):
        dataset          = bigquery.Dataset(dataset_ref)
        dataset.location = 'US'
        client_bq.create_dataset(dataset)
        print(f"Dataset '{DATASET_ID}' criado com sucesso.")
    else:
        raise e

In [ ]:
# Define o schema da tabela de destino
schema = [
    bigquery.SchemaField('order_id',                      'STRING'),
    bigquery.SchemaField('order_estimated_delivery_date', 'TIMESTAMP'),
    bigquery.SchemaField('order_delivered_customer_date', 'TIMESTAMP'),
    bigquery.SchemaField('atraso_dias',                   'INTEGER'),   # ✅ nome corrigido (não é uma média)
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition='WRITE_APPEND'  # Adiciona aos dados existentes sem sobrescrever
)

job = client_bq.load_table_from_dataframe(df_atraso, TABLE_PATH, job_config=job_config)
job.result()

print(f"Tabela '{TABLE_PATH}' carregada com sucesso!")

## ⬆️ 8. Upload do Resultado de Volta ao Cloud Storage

Exportando o DataFrame como CSV e fazendo o upload pro bucket — fechando o ciclo: buscamos dado da nuvem, processamos, e devolvemos pra nuvem.

In [ ]:
CSV_LOCAL   = 'dados.csv'
CSV_DESTINO = 'dados/dados.csv'

# Salva o DataFrame como CSV localmente
df_atraso.to_csv(CSV_LOCAL, index=False)

# Faz o upload pro bucket
bucket = client_gcs.bucket(BUCKET_NAME)  # ✅ método atual (get_bucket é depreciado)
blob   = bucket.blob(CSV_DESTINO)
blob.upload_from_filename(CSV_LOCAL)

print(f"Arquivo '{CSV_LOCAL}' enviado para gs://{BUCKET_NAME}/{CSV_DESTINO}")